import pandas as pd
from datetime import datetime

# Load orders table — dates come in as plain strings like "2018-05-11 20:07:00.000"
orders = pd.read_csv("https://cdn.enqurious.com/documents/ca1a31d6-b8c9-4f71-8346-36989a1de15f_exorders.csv")

# Define the raw format that matches what's actually stored in the CSV
# "2018-05-11 20:07:00.000" → %Y-%m-%d %H:%M:%S.%f
raw_fmt    = "%Y-%m-%d %H:%M:%S.%f"

# Define the target output format for the operations digest label
# "Friday, 11 May 2018" → %A, %d %B %Y
output_fmt = "%A, %d %B %Y"

# SQL equivalent (PostgreSQL):
#   SELECT TO_CHAR(order_purchase_date, 'Day, DD Month YYYY') AS purchase_label FROM orders;
#
# Python: strptime parses the string → datetime, then strftime formats it → label
# We use .apply() to run this two-step transformation on every row of the column
orders["purchase_label"] = orders["order_purchase_date"].apply(
    lambda val: datetime.strptime(val, raw_fmt).strftime(output_fmt)
)

# Select only the columns needed for the digest report
result = orders[["order_id", "order_purchase_date", "purchase_label"]]

print(result.head())

import pandas as pd
from datetime import datetime

# Load orders table
orders = pd.read_csv("https://cdn.enqurious.com/documents/ca1a31d6-b8c9-4f71-8346-36989a1de15f_exorders.csv")

raw_fmt        = "%Y-%m-%d %H:%M:%S.%f"
digest_fmt     = "%A, %d %B %Y"
time_label_fmt = "%I:%M %p"   # %I = 12-hour clock (01-12), %p = AM or PM

# Helper function: parse once → produce TWO labels in one pass
# WHY return pd.Series? When .apply() sees a pd.Series return, pandas automatically
# expands each key into a separate column. No need to call .apply() twice.
def build_labels(val):
    dt = datetime.strptime(val, raw_fmt)
    return pd.Series({
        "purchase_label"    : dt.strftime(digest_fmt),
        "time_of_day_label" : dt.strftime(time_label_fmt)
    })

# One .apply() call → two new columns created simultaneously
labels = orders["order_purchase_date"].apply(build_labels)

# Concatenate the new label columns next to the original identifiers
result = pd.concat([orders[["order_id", "order_purchase_date"]], labels], axis=1)

print(result.head())

import pandas as pd
from datetime import datetime

# Load customers table — joining_date is a simple date string (no time component)
customers = pd.read_csv("https://cdn.enqurious.com/documents/7b9c870b-6634-493a-8816-6e863c3505d0_excustomers.csv")

# Raw format: "2017-08-22" (date only — no hours/minutes/seconds)
# Output:     "Aug-2017" (%b = abbreviated month name, %Y = 4-digit year)
raw_fmt    = "%Y-%m-%d"
output_fmt = "%b-%Y"

# SQL equivalent: TO_CHAR(joining_date, 'Mon-YYYY') AS joining_cohort
# Parse the date string → datetime object, then format it → cohort label
customers["joining_cohort"] = customers["joining_date"].apply(
    lambda val: datetime.strptime(val, raw_fmt).strftime(output_fmt)
)

# Select the relevant columns for the CRM cohort report
result = customers[["customer_id", "customer_name", "joining_date", "joining_cohort"]]

print(result.head())

import pandas as pd
from datetime import datetime

# Load campaigns table — start_date and end_date are simple date strings
campaigns = pd.read_csv("https://cdn.enqurious.com/documents/fb6f45f8-91cc-4b68-8de8-2e2e97734839_excampaigns.csv")

# Both columns use the same raw format and the same target output format
raw_fmt    = "%Y-%m-%d"
output_fmt = "%d/%m/%Y"   # DD/MM/YYYY (day first — international format, NOT US MM/DD)

# WHY a named helper instead of a lambda?
# The same transformation applies to TWO columns — define the logic once, reuse twice
# Lambda would duplicate the expression; a named function keeps it DRY (Don't Repeat Yourself)
def reformat_date(val):
    # SQL equivalent: TO_CHAR(date_col, 'DD/MM/YYYY')
    return datetime.strptime(val, raw_fmt).strftime(output_fmt)

# Apply the same helper to both campaign date columns
campaigns["start_label"] = campaigns["start_date"].apply(reformat_date)
campaigns["end_label"]   = campaigns["end_date"].apply(reformat_date)

# Select the relevant columns for the marketing report
result = campaigns[["campaign_id", "campaign_type", "start_date", "end_date",
                     "start_label", "end_label"]]

print(result.head())

In [10]:
import pandas as pd
from datetime import datetime

# Load table
orders = pd.read_csv("https://cdn.enqurious.com/documents/ca1a31d6-b8c9-4f71-8346-36989a1de15f_exorders.csv")

# Define the raw format of order_purchase_date
raw_fmt    = "%Y-%m-%d %H:%M:%S.%f"  # includes microseconds if present
# Define the output format for the operations digest label
output_fmt = "%A, %d %B %Y"

# Parse the raw string → datetime, then format → label string
orders["purchase_label"] = orders["order_purchase_date"].apply(
    lambda val: datetime.strptime(val, raw_fmt).strftime(output_fmt)
)

# Select only the columns needed for the digest report
result = orders[["order_id", "order_purchase_date", "purchase_label"]]

print(result.head())

         order_id      order_purchase_date              purchase_label
0  CA-2014-100006  2018-05-11 20:07:00.000         Friday, 11 May 2018
1  CA-2014-100090  2018-01-30 10:21:00.000    Tuesday, 30 January 2018
2  CA-2014-100293  2017-10-04 14:15:00.000  Wednesday, 04 October 2017
3  CA-2014-100328  2018-05-17 14:22:00.000       Thursday, 17 May 2018
4  CA-2014-100363  2017-08-28 17:01:00.000      Monday, 28 August 2017


In [11]:
import pandas as pd
from datetime import datetime

# Load table
orders = pd.read_csv("https://cdn.enqurious.com/documents/ca1a31d6-b8c9-4f71-8346-36989a1de15f_exorders.csv")

raw_fmt        = "%Y-%m-%d %H:%M:%S.%f"
digest_fmt     = "%A, %d %B %Y"
time_label_fmt = "%I:%M %p"   # 12-hour clock with AM/PM

# Helper: parse once, produce both labels
def build_labels(val):
    dt = datetime.strptime(val, raw_fmt)
    return pd.Series({
        "purchase_label"    : dt.strftime(digest_fmt),
        "time_of_day_label" : dt.strftime(time_label_fmt)
    })

labels = orders["order_purchase_date"].apply(build_labels)

# Concatenate labels back to the original DataFrame
result = pd.concat([orders[["order_id", "order_purchase_date"]], labels], axis=1)

print(result.head())

         order_id      order_purchase_date              purchase_label  \
0  CA-2014-100006  2018-05-11 20:07:00.000         Friday, 11 May 2018   
1  CA-2014-100090  2018-01-30 10:21:00.000    Tuesday, 30 January 2018   
2  CA-2014-100293  2017-10-04 14:15:00.000  Wednesday, 04 October 2017   
3  CA-2014-100328  2018-05-17 14:22:00.000       Thursday, 17 May 2018   
4  CA-2014-100363  2017-08-28 17:01:00.000      Monday, 28 August 2017   

  time_of_day_label  
0          08:07 PM  
1          10:21 AM  
2          02:15 PM  
3          02:22 PM  
4          05:01 PM  


In [12]:
import pandas as pd
from datetime import datetime

# Load table
customers = pd.read_csv("https://cdn.enqurious.com/documents/7b9c870b-6634-493a-8816-6e863c3505d0_excustomers.csv")

# Define the raw format of joining_date (date only — no time component)
raw_fmt    = "%Y-%m-%d"
# Define the output format for the CRM cohort label
output_fmt = "%b-%Y"

# Parse and format in a single .apply() call
customers["joining_cohort"] = customers["joining_date"].apply(
    lambda val: datetime.strptime(val, raw_fmt).strftime(output_fmt)
)

# Select the relevant columns for the CRM report
result = customers[["customer_id", "customer_name", "joining_date", "joining_cohort"]]

print(result.head())

  customer_id    customer_name joining_date joining_cohort
0    AA-10375     Allen Armold   2017-08-22       Aug-2017
1    AA-10480     Andrew Allen   2017-02-15       Feb-2017
2    AB-10060  Adam Bellavance   2017-04-16       Apr-2017
3    AB-10165      Alan Barnes   2017-01-13       Jan-2017
4    AC-10420    Alyssa Crouse   2017-01-27       Jan-2017


In [13]:
import pandas as pd
from datetime import datetime

# Load table
campaigns = pd.read_csv("https://cdn.enqurious.com/documents/fb6f45f8-91cc-4b68-8de8-2e2e97734839_excampaigns.csv")

# Define formats once — raw input and desired output
raw_fmt    = "%Y-%m-%d"
output_fmt = "%d/%m/%Y"

# Reusable helper: parse then format
def reformat_date(val):
    return datetime.strptime(val, raw_fmt).strftime(output_fmt)

# Apply to both campaign date columns
campaigns["start_label"] = campaigns["start_date"].apply(reformat_date)
campaigns["end_label"]   = campaigns["end_date"].apply(reformat_date)

# Select columns for the marketing report
result = campaigns[["campaign_id", "campaign_type", "start_date", "end_date",
                     "start_label", "end_label"]]

print(result.head())

   campaign_id              campaign_type  start_date    end_date start_label  \
0         1481           Loyalty Campaign  2017-10-09  2018-06-09  09/10/2017   
1          366  First-Time Buyer Campaign  2017-12-12  2018-03-13  12/12/2017   
2          656  First-Time Buyer Campaign  2018-04-03  2018-06-08  03/04/2018   
3          856                Retargeting  2017-03-28  2017-10-09  28/03/2017   
4          136                   Discount  2017-06-18  2017-09-14  18/06/2017   

    end_label  
0  09/06/2018  
1  13/03/2018  
2  08/06/2018  
3  09/10/2017  
4  14/09/2017  
